# 05 - Like Games Grouping

Edit the Python inputs in the next cell and run all cells in order. All settings come from code.

- `USER_EXPERIMENT_NAME` labels the run; its text does not choose a parent process or feature weights.
- `PARENT_GAME_PROCESS` selects parent handling: P1 restricts candidates with parents for targets with parents; P2 uses normal similarity; P3 puts an available, earlier commercial parent first in sales/rental lists.
- `FEATURE_GROUP_WEIGHTS` sets similarity feature importance. Both cosine and Euclidean distance are supported.
- Scaling uses only source games released before each target. Missing numeric features become zero.
- Normal neighbors must satisfy the release cutoff, minimum age gap and source mapping requirements. P3 parent insertion has its own eligibility rules.

## Saving

`RESULT_WRITE_MODE = "overwrite"` is enabled temporarily. **The final cell replaces every row in `development_042_silver_sandbox.demand_forecast.like_games_ai_attr` with this generation.** No table is changed until that cell runs. Previous MLflow runs are retained.

Set `RESULT_WRITE_MODE = "append"` to preserve multiple generations. Append retries for the same generation avoid duplicate rows. Rerun setup and generation after changing settings; saving rejects stale results.

The final cell logs the actual settings, source Delta version, preprocessing, prepared input and neighbors to `/Shared/AI_Feature_Extraction_Similarity`, writes the Delta table, then verifies the saved generation.


In [0]:
# Experiment inputs: edit these, then run the notebook from the top.
# These Python variables are the only configuration inputs.
USER_EXPERIMENT_NAME = "p3_mech_theme"
PARENT_GAME_PROCESS = "p3_include_parent"
# Choices: p1_parent_games_only, p2_current, p3_include_parent.

FEATURE_GROUP_WEIGHTS = {
    "mechanics": 2.0,
    "themes": 2.0,
    "reels": 1.0,
    "persistence": 2.0,
    "ways": 1.0,
    "months_since_release": 0.0,
}
DISTANCE_METHOD = "cosine"  # cosine or euclidean; both are implemented.
N_SALES_NEIGHBORS = 5
N_RENTAL_NEIGHBORS = 5
N_EILERS_NEIGHBORS = 5

# One MLflow experiment contains all similarity-generation runs.
# Use a workspace experiment path you can write to.
MLFLOW_EXPERIMENT_PATH = "/Shared/AI_Feature_Extraction_Similarity"
IMPLEMENTATION_VERSION = "05_similarity_v3_no_widgets_optional_overwrite"

# TEMPORARY: replaces ALL rows in like_games_dest when the final cell runs.
# Change to "append" to preserve multiple experiments again.
RESULT_WRITE_MODE = "overwrite"

USER_EXPERIMENT_NAME = (USER_EXPERIMENT_NAME or "").strip()
PARENT_GAME_PROCESS = PARENT_GAME_PROCESS.strip().lower()
if PARENT_GAME_PROCESS not in {
    "p1_parent_games_only", "p2_current", "p3_include_parent"
}:
    raise ValueError(f"Unknown parent-game process: {PARENT_GAME_PROCESS!r}")

In [0]:
ai_attr_silver_path = "development_042_silver_sandbox.demand_forecast.game_attr_ai_silver"
like_games_dest = "development_042_silver_sandbox.demand_forecast.like_games_ai_attr"

# None resolves the latest Delta version ONCE; an integer replays that snapshot.
SOURCE_TABLE_VERSION = None
MAX_NEIGHBOUR_DATE = "2021-06-01"  # Neighbors must be released strictly after this date.
MIN_AGE_GAP_MONTHS = 12

# Historical scaling: each target uses only games released before that target.
# Missing numeric features are zero. Missing release dates stay missing.
# The final cell uses RESULT_WRITE_MODE: overwrite replaces the table; append keeps history.

In [0]:
from pyspark.sql import functions as F

def quoted_table_name(name):
    return ".".join("`" + part.replace("`", "``") + "`" for part in name.split("."))

if SOURCE_TABLE_VERSION is not None and (
    isinstance(SOURCE_TABLE_VERSION, bool)
    or not isinstance(SOURCE_TABLE_VERSION, int)
    or SOURCE_TABLE_VERSION < 0
):
    raise ValueError("SOURCE_TABLE_VERSION must be None or a nonnegative integer.")

source_history = spark.sql(f"DESCRIBE HISTORY {quoted_table_name(ai_attr_silver_path)}")
RESOLVED_SOURCE_VERSION = (
    int(source_history.agg(F.max("version").alias("v")).first()["v"])
    if SOURCE_TABLE_VERSION is None else int(SOURCE_TABLE_VERSION)
)
SOURCE_LOAD_SETTINGS = (ai_attr_silver_path, SOURCE_TABLE_VERSION)
ai_attr_silver = (
    spark.read.option("versionAsOf", RESOLVED_SOURCE_VERSION).table(ai_attr_silver_path)
)
print(f"Pinned source: {ai_attr_silver_path}, Delta version {RESOLVED_SOURCE_VERSION}")
display(ai_attr_silver)

In [0]:
import re
import uuid
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, NumericType, StringType


# Configuration validation and a fresh generation ID.
import copy
import hashlib
import json
import math
from numbers import Real, Integral

def similarity_settings():
    """Capture public inputs; changing any requires rerunning setup and generation."""
    return {
        "raw_weights": copy.deepcopy(FEATURE_GROUP_WEIGHTS),
        "distance_method": DISTANCE_METHOD.strip().lower(),
        "parent_game_process": PARENT_GAME_PROCESS,
        "neighbor_limits": {
            "sales": N_SALES_NEIGHBORS, "rental": N_RENTAL_NEIGHBORS,
            "eilers": N_EILERS_NEIGHBORS,
        },
        "neighbor_released_after": str(MAX_NEIGHBOUR_DATE),
        "minimum_age_gap_months": MIN_AGE_GAP_MONTHS,
        "source_table": ai_attr_silver_path,
        "source_version": RESOLVED_SOURCE_VERSION,
        "source_version_requested": SOURCE_TABLE_VERSION,
        "destination_table": like_games_dest,
        "result_write_mode": RESULT_WRITE_MODE,
        "mlflow_experiment_path": MLFLOW_EXPERIMENT_PATH,
        "implementation_version": IMPLEMENTATION_VERSION,
        "user_experiment_name": USER_EXPERIMENT_NAME,
    }

if SOURCE_LOAD_SETTINGS != (ai_attr_silver_path, SOURCE_TABLE_VERSION):
    raise ValueError("Source settings changed. Rerun the source-loading cell first.")

method = DISTANCE_METHOD.strip().lower()
if method not in {"cosine", "euclidean"}:
    raise ValueError("DISTANCE_METHOD must be 'cosine' or 'euclidean'.")
if PARENT_GAME_PROCESS not in {
    "p1_parent_games_only", "p2_current", "p3_include_parent"
}:
    raise ValueError("Unknown PARENT_GAME_PROCESS.")
for key, value in {
    "N_SALES_NEIGHBORS": N_SALES_NEIGHBORS,
    "N_RENTAL_NEIGHBORS": N_RENTAL_NEIGHBORS,
    "N_EILERS_NEIGHBORS": N_EILERS_NEIGHBORS,
    "MIN_AGE_GAP_MONTHS": MIN_AGE_GAP_MONTHS,
}.items():
    if isinstance(value, bool) or not isinstance(value, Integral) or value < 0:
        raise ValueError(f"{key} must be a nonnegative integer.")
if pd.isna(pd.Timestamp(MAX_NEIGHBOUR_DATE)):
    raise ValueError("MAX_NEIGHBOUR_DATE must be a valid date.")
if not MLFLOW_EXPERIMENT_PATH.startswith("/"):
    raise ValueError("MLFLOW_EXPERIMENT_PATH must be an absolute workspace experiment path.")

if RESULT_WRITE_MODE not in {"append", "overwrite"}:
    raise ValueError("RESULT_WRITE_MODE must be append or overwrite.")

PREPARED_SETTINGS = similarity_settings()
EXPERIMENT_ID = f"aiattr_{uuid.uuid4().hex}"
SIMILARITY_MLFLOW_RUN_ID = None
SAVE_IDENTITY = None

# -----------------------------
# Inspect source schema
# -----------------------------

source_df = ai_attr_silver
source_columns = {field.name: field for field in source_df.schema.fields}

OWNED_STATUS_COL = (
    "own_status" if "own_status" in source_columns
    else "owned_status" if "owned_status" in source_columns
    else None
)

required_columns = [
    "game_name",
    "parent_game",
    "to_sales_name_match",
    "to_rental_name_match",
    "game_release_date",
    "game_category",
    "join_game_name_key",
    "parent_own_status",
    "game_classification",
    "max_months_since_release",
]

if OWNED_STATUS_COL is None:
    raise ValueError("Could not find own_status or owned_status in ai_attr_silver.")

missing_columns = [c for c in required_columns if c not in source_columns]
if missing_columns:
    raise ValueError(
        f"Missing required columns in ai_attr_silver: {missing_columns}"
    )

for flag_col in ["has_sales_match", "has_rental_match", "has_eilers_ts_match"]:
    if flag_col not in source_columns:
        raise ValueError(f"Missing required match flag: {flag_col}")


# -----------------------------
# Helpers for extracting numeric values
# -----------------------------

def extract_numeric_from_string(value_expr):
    """Extract the first integer or decimal from a string; return null if absent."""
    extracted = F.regexp_extract(
        value_expr.cast("string"),
        r"(-?\d+(?:\.\d+)?)",
        1,
    )

    return F.when(
        F.length(extracted) > 0,
        extracted.cast("double"),
    ).otherwise(F.lit(None).cast("double"))


def get_leaf_fields(data_type, prefix=()):
    """Return scalar leaf paths for fields inside nested structs."""
    leaves = []

    if isinstance(data_type, StructType):
        for field in data_type.fields:
            leaves.extend(
                get_leaf_fields(field.dataType, prefix + (field.name,))
            )
    else:
        leaves.append((prefix, data_type))

    return leaves


def numeric_feature_expr(root_col, preferred_names):
    """Extract a numeric feature from a numeric, string, or struct column."""
    data_type = source_columns[root_col].dataType

    if isinstance(data_type, NumericType):
        return F.col(root_col).cast("double")

    if isinstance(data_type, StringType):
        return extract_numeric_from_string(F.col(root_col))

    if isinstance(data_type, StructType):
        leaves = get_leaf_fields(data_type)
        chosen = None

        # Prefer fields with names that commonly contain the extracted value.
        for preferred_name in preferred_names:
            chosen = next(
                (
                    (path, leaf_type)
                    for path, leaf_type in leaves
                    if path[-1].lower() == preferred_name
                    and isinstance(leaf_type, (NumericType, StringType))
                ),
                None,
            )
            if chosen:
                break

        # Otherwise use the first numeric leaf, then the first string leaf.
        if chosen is None:
            chosen = next(
                (
                    (path, leaf_type)
                    for path, leaf_type in leaves
                    if isinstance(leaf_type, NumericType)
                ),
                None,
            )

        if chosen is None:
            chosen = next(
                (
                    (path, leaf_type)
                    for path, leaf_type in leaves
                    if isinstance(leaf_type, StringType)
                ),
                None,
            )

        if chosen is None:
            source_df.select(root_col).printSchema()
            raise ValueError(
                f"Could not find a numeric or string field inside {root_col}."
            )

        path, leaf_type = chosen
        value_expr = F.col(root_col)

        for part in path:
            value_expr = value_expr.getField(part)

        print(f"Extracting {root_col} from nested field: {'.'.join(path)}")

        if isinstance(leaf_type, NumericType):
            return value_expr.cast("double")

        return extract_numeric_from_string(value_expr)

    source_df.select(root_col).printSchema()
    raise ValueError(f"Unsupported type for {root_col}: {data_type}")


# -----------------------------
# Resolve reel count
# -----------------------------

reel_root_col = next(
    (c for c in ["number_of_reels", "reels"] if c in source_columns),
    None,
)

if reel_root_col is None:
    raise ValueError("Could not find number_of_reels or reels.")

reel_expr = numeric_feature_expr(
    reel_root_col,
    [
        "number_of_reels",
        "reel_count",
        "count",
        "base_reels",
        "reels",
        "value",
        "score",
    ],
)


# -----------------------------
# Identify theme and mechanic features
# -----------------------------

theme_cols = [
    c for c in source_columns
    if c.startswith("theme_")
    and c.endswith("_score")
    and c != "theme_presence_score"
]

mechanic_present_cols = [
    c for c in source_columns
    if c.startswith("mechanic_") and c.endswith("_present")
]
mechanic_score_cols = [
    c for c in source_columns
    if c.startswith("mechanic_") and c.endswith("_score")
]

# Prefer binary mechanic presence features when available.
mechanic_cols = mechanic_present_cols or mechanic_score_cols

if not theme_cols:
    raise ValueError("No theme_*_score columns found.")
if not mechanic_cols:
    raise ValueError(
        "No mechanic_*_present or mechanic_*_score columns found."
    )


# -----------------------------
# Resolve persistence feature(s)
# -----------------------------

persistence_candidates = [
    "persistence_score",
    "perceived_persistence_score",
    "persistence",
    "perceived_persistence",
]
persistence_cols = [
    c for c in persistence_candidates if c in source_columns
]

if not persistence_cols:
    raise ValueError("No recognized persistence feature found.")

persistence_exprs = {
    c: numeric_feature_expr(
        c,
        ["value", "score", "persistence", "persistence_score"],
    )
    for c in persistence_cols
}


# -----------------------------
# Resolve ways/paylines feature
# -----------------------------

ways_candidates = [
    "ways",
    "lines_or_ways",
    "paylines_or_ways_count",
    "lines",
]
ways_col = next(
    (c for c in ways_candidates if c in source_columns),
    None,
)

if ways_col is None:
    raise ValueError("No recognized ways/lines feature found.")

ways_expr = numeric_feature_expr(
    ways_col,
    [
        "value",
        "ways",
        "lines",
        "count",
        "paylines_or_ways_count",
        "score",
    ],
)


# -----------------------------
# Select source and matched-name columns
# -----------------------------

sales_name_col = (
    "to_sales_name_match"
    if "to_sales_name_match" in source_columns
    else "game_name"
)
rental_name_col = "to_rental_name_match"
eilers_name_col = (
    "to_eilers_ts_name_match"
    if "to_eilers_ts_name_match" in source_columns
    else "game_name"
)

selected_cols = [
    "game_name",
    "parent_game",
    "game_category",
    "join_game_name_key",
    OWNED_STATUS_COL,
    "parent_own_status",
    "game_classification",
    "max_months_since_release",
    'game_release_date',
    "has_sales_match",
    "has_rental_match",
    "has_eilers_ts_match",
    rental_name_col,
]

if sales_name_col != "game_name":
    selected_cols.append(sales_name_col)

if eilers_name_col != "game_name":
    selected_cols.append(eilers_name_col)


# -----------------------------
# Clean and unpack features
# -----------------------------

clean_df = source_df.select(
    *[F.col(c) for c in selected_cols],
    reel_expr.alias("reels_clean"),
    ways_expr.alias("ways_clean"),
    *[
        persistence_exprs[c].alias(c)
        for c in persistence_cols
    ],
    *[
        F.expr(f"try_cast(`{c}` AS DOUBLE)").alias(c)
        for c in theme_cols
    ],
    *[
        F.expr(f"try_cast(`{c}` AS DOUBLE)").alias(c)
        for c in mechanic_cols
    ],
)

# Standardize the ownership column name for the next cell.
if OWNED_STATUS_COL != "own_status":
    clean_df = clean_df.withColumnRenamed(
        OWNED_STATUS_COL,
        "own_status",
    )

# Fill missing theme and mechanic values with zero.
for c in theme_cols + mechanic_cols:
    clean_df = clean_df.withColumn(
        c,
        F.coalesce(F.col(c), F.lit(0.0)),
    )

# Normalize descriptive ownership metadata. Null or blank values use "__unknown__".
grouping_cols = [
    "parent_own_status",
]

for c in grouping_cols:
    clean_df = clean_df.withColumn(
        c,
        F.when(
            F.col(c).isNull()
            | (F.trim(F.col(c).cast("string")) == ""),
            F.lit("__unknown__"),
        ).otherwise(
            F.lower(F.trim(F.col(c).cast("string")))
        ),
    )

# Prepare the canonical game key; merge duplicate games below.
clean_df = (
    clean_df
    .withColumn(
        "_game_name_key",
        F.regexp_replace(F.lower(F.trim(F.col("join_game_name_key"))), r"\s+", " "),
    )
    .where(F.col("_game_name_key").isNotNull() & (F.col("_game_name_key") != ""))
)

pdf = clean_df.toPandas()

if len(pdf) < 2:
    raise ValueError("Fewer than two games remain after preprocessing.")


# -----------------------------
# Clean flags and numeric values
# -----------------------------

def as_bool(value):
    if pd.isna(value):
        return False
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


for c in ["has_sales_match", "has_rental_match", "has_eilers_ts_match"]:
    pdf[c] = pdf[c].map(as_bool)

# One row per canonical game, even if several overview documents exist.
# Keep one document's features and combine source matches across documents.
def normalize_game_reference(value):
    """Normalize exact game references; null-like strings are missing."""
    if value is None or pd.isna(value):
        return None
    key = " ".join(str(value).strip().lower().split())
    return None if key in {
        "", "none", "null", "<na>", "nan", "na", "n/a", "__unknown__"
    } else key


def deduplicate_games(frame):
    frame = frame.copy()
    frame["join_game_name_key"] = (
        frame["join_game_name_key"].astype("string").str.strip().str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )
    frame = frame.loc[
        frame["join_game_name_key"].notna() & frame["join_game_name_key"].ne("")
    ]
    # Prefer the alphabetically first document name as the feature row.
    # Resolve equal document names deterministically, regardless of Spark row order.
    frame["_document_tiebreak"] = [
        hashlib.sha256(json.dumps(record, sort_keys=True).encode()).hexdigest()
        for record in json.loads(frame.to_json(orient="records", date_format="iso"))
    ]
    frame = frame.sort_values(["game_name", "_document_tiebreak"], kind="stable", na_position="last")
    match_columns = [
        ("has_sales_match", sales_name_col),
        ("has_rental_match", rental_name_col),
        ("has_eilers_ts_match", eilers_name_col),
    ]
    merged = []
    for _, group in frame.groupby("join_game_name_key", sort=False):
        row = group.iloc[0].drop(labels=["_document_tiebreak"]).copy()
        # Retain a parent supplied by another document when the feature row
        # has no parent. Use the same stable document order as source matches.
        parent_names = group["parent_game"].map(normalize_game_reference).dropna()
        row["parent_game"] = None if parent_names.empty else parent_names.iloc[0]
        for flag_col, name_col in match_columns:
            row[flag_col] = bool(group[flag_col].any())
            names = group.loc[group[flag_col], name_col].astype("string").str.strip()
            usable = names.notna() & ~names.str.lower().isin(
                ["", "none", "null", "<na>", "nan", "na", "n/a"]
            )
            names = names.loc[usable]
            # A false match flag has no usable mapping for this source.
            row[name_col] = None if names.empty else str(names.iloc[0])
        merged.append(row)
    if not merged:
        return frame.iloc[:0].reset_index(drop=True)
    return pd.DataFrame(merged).reset_index(drop=True)


before_dedup = len(pdf)
pdf = deduplicate_games(pdf)
print(f"Merged {before_dedup - len(pdf)} duplicate game rows by join_game_name_key.")

feature_groups = {
    "mechanics": mechanic_cols,
    "themes": theme_cols,
    "reels": ["reels_clean"],
    "persistence": persistence_cols,
    "ways": ["ways_clean"],
    "months_since_release": ["max_months_since_release"],
}

numeric_cols = [
    c
    for columns in feature_groups.values()
    for c in columns
]

# Keep null release ages null for the earlier-than comparison.
release_age = pd.to_numeric(
    pdf["max_months_since_release"],
    errors="coerce",
)

# Coerce malformed values to null, then fill feature nulls with zero.
for c in numeric_cols:
    pdf[c] = pd.to_numeric(
        pdf[c],
        errors="coerce",
    ).replace([np.inf, -np.inf], np.nan).fillna(0.0)

pdf["max_months_since_release"] = release_age


# -----------------------------
# Normalize feature-group weights
# -----------------------------

unknown_groups = sorted(set(FEATURE_GROUP_WEIGHTS) - set(feature_groups))
if unknown_groups:
    raise ValueError(f"Unknown feature groups: {unknown_groups}")
invalid_weights = {
    group: weight for group, weight in FEATURE_GROUP_WEIGHTS.items()
    if isinstance(weight, bool) or not isinstance(weight, Real)
    or not math.isfinite(float(weight)) or weight < 0
}
if invalid_weights:
    raise ValueError(f"Feature weights must be finite nonnegative numbers: {invalid_weights}")

active_groups = {
    group: columns
    for group, columns in feature_groups.items()
    if columns and FEATURE_GROUP_WEIGHTS.get(group, 0) > 0
}

weight_total = sum(
    FEATURE_GROUP_WEIGHTS[group]
    for group in active_groups
)

if not math.isfinite(float(weight_total)) or weight_total <= 0:
    raise ValueError("At least one feature group must have a positive weight.")

normalized_weights = {
    group: FEATURE_GROUP_WEIGHTS[group] / weight_total
    for group in active_groups
}


# -----------------------------
# Generate experiment name
# -----------------------------

def readable_slug(text, max_length=70):
    text = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        str(text),
    ).strip("_").lower()

    return text[:max_length].strip("_") or "ai_attribute_similarity"



EXPERIMENT_READABLE_NAME = USER_EXPERIMENT_NAME or (
    f"{PARENT_GAME_PROCESS}_{method}_" + "_".join(
        f"{group}{float(FEATURE_GROUP_WEIGHTS.get(group, 0)):g}"
        for group in feature_groups
    )
)

EXPERIMENT_READABLE_NAME = readable_slug(
    EXPERIMENT_READABLE_NAME
)

print(f"Experiment name: {EXPERIMENT_READABLE_NAME}")
print(f"Experiment ID: {EXPERIMENT_ID}")
print(f"Parent game process: {PARENT_GAME_PROCESS}")
print(f"Distance method: {method}")
print(f"Normalized feature weights: {normalized_weights}")
print(f"Games prepared: {len(pdf)}")
print(f"Result write mode: {RESULT_WRITE_MODE}; destination: {like_games_dest}")

In [0]:
import numpy as np
import pandas as pd

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    DoubleType,
    ArrayType,
    MapType,
    DateType,
)


# Prepare the immutable input snapshot for this generation.
if similarity_settings() != PREPARED_SETTINGS:
    raise ValueError("Settings changed after preparation. Rerun setup before generation.")
pdf = pdf.reset_index(drop=True)
feature_columns = list(dict.fromkeys(
    column for columns in active_groups.values() for column in columns
))
required_columns = [
    "join_game_name_key", "parent_game", "game_release_date", "max_months_since_release",
    "has_sales_match", "has_rental_match", "has_eilers_ts_match",
    sales_name_col, rental_name_col, eilers_name_col, *feature_columns,
]
missing_columns = sorted(set(required_columns) - set(pdf.columns))
if missing_columns:
    raise ValueError(f"Missing required columns in pdf: {missing_columns}")

raw_features = (
    pdf[feature_columns].apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan).fillna(0.0)
)
raw_feature_matrix = raw_features.to_numpy(dtype=np.float64, copy=True)
column_indices = {column: index for index, column in enumerate(feature_columns)}
feature_multipliers = np.ones(len(feature_columns), dtype=np.float64)
for group, columns in active_groups.items():
    feature_multipliers[[column_indices[column] for column in columns]] = (
        np.sqrt(float(normalized_weights[group]) / len(columns))
    )
historical_scaler_states = {}
_last_scaler_key = None
_last_weighted_matrix = None

game_records = pdf.to_dict("records")
game_names = pdf["join_game_name_key"].fillna("").astype(str).to_numpy()

candidate_ages = pd.to_numeric(
    pdf["max_months_since_release"],
    errors="coerce",
).to_numpy(dtype=np.float64, na_value=np.nan)
valid_ages = ~np.isnan(candidate_ages)

candidate_release_dates = pd.to_datetime(
    pdf["game_release_date"],
    errors="coerce",
).to_numpy(dtype="datetime64[ns]")


# --------------------------------------------------
# Match-flag and name helpers
# --------------------------------------------------
def has_valid_name(value):
    """Return whether a value is a usable matched source name."""
    if pd.isna(value):
        return False

    return str(value).strip().lower() not in {
        "",
        "none",
        "null",
        "<na>",
        "nan",
        "na",
        "n/a",
    }


def to_bool(value):
    """Safely interpret boolean or string match flags."""
    if pd.isna(value):
        return False

    if isinstance(value, str):
        return value.strip().lower() in {"true", "1", "yes", "y"}

    return bool(value)


def optional_string(record, column):
    """Return a string value when present; otherwise return None."""
    value = record.get(column)

    if value is None or pd.isna(value):
        return None

    return str(value)


has_sales_match = (
    pdf["has_sales_match"]
    .map(to_bool)
    .to_numpy(dtype=bool)
)
has_rental_match = (
    pdf["has_rental_match"]
    .map(to_bool)
    .to_numpy(dtype=bool)
)
has_eilers_match = (
    pdf["has_eilers_ts_match"]
    .map(to_bool)
    .to_numpy(dtype=bool)
)

valid_sales_names = (
    pdf["to_sales_name_match"].map(has_valid_name).to_numpy(dtype=bool)
    & pdf[sales_name_col].map(has_valid_name).to_numpy(dtype=bool)
)
valid_rental_names = (
    pdf[rental_name_col].map(has_valid_name).to_numpy(dtype=bool)
)
valid_eilers_names = (
    pdf[eilers_name_col].map(has_valid_name).to_numpy(dtype=bool)
)


# Final target eligibility is based on the source match flags.
# Keep all source rows in pdf so existing scaling and Eilers candidate
# coverage are preserved. Only eligible targets become output rows.
eligible_targets = (
    has_eilers_match
    & (has_sales_match | has_rental_match)
)


def sales_rental_type(index):
    if has_sales_match[index] and has_rental_match[index]:
        return "both"
    if has_sales_match[index]:
        return "sales"
    if has_rental_match[index]:
        return "rental"
    return None



# --------------------------------------------------
# Resolve parent references without fuzzy matching
# --------------------------------------------------
parent_keys = pdf["parent_game"].map(normalize_game_reference).to_numpy()
has_parent_game = np.array([key is not None for key in parent_keys], dtype=bool)
canonical_lookup = {
    normalize_game_reference(name): index
    for index, name in enumerate(game_names)
}

# Prefer the canonical key. Otherwise accept a unique exact matched-name
# alias; ambiguous aliases are reported and never guessed.
parent_alias_lookup = {}
for index, record in enumerate(game_records):
    for column in dict.fromkeys([sales_name_col, rental_name_col, eilers_name_col]):
        alias = normalize_game_reference(record.get(column))
        if alias is not None:
            parent_alias_lookup.setdefault(alias, set()).add(index)


def resolve_parent_index(target_idx):
    parent_key = parent_keys[target_idx]
    if parent_key is None:
        return None, "no_parent"
    if parent_key in canonical_lookup:
        parent_idx = canonical_lookup[parent_key]
    else:
        matches = parent_alias_lookup.get(parent_key, set())
        if not matches:
            return None, "not_found"
        if len(matches) != 1:
            return None, "ambiguous_name"
        parent_idx = next(iter(matches))
    if parent_idx == target_idx:
        return None, "self_reference"
    return int(parent_idx), "resolved"


parent_resolutions = [resolve_parent_index(i) for i in range(len(game_records))]


# Fit scaling independently for each prediction date using earlier releases.
# This keeps future games out of historical min/max bounds. Only one full
# transformed matrix is cached; small fitted states are saved for every date.
def weighted_matrix_for(target_idx):
    global _last_scaler_key, _last_weighted_matrix
    target_date = candidate_release_dates[target_idx]
    if np.isnat(target_date):
        raise ValueError("A target needs a release date for historical scaling.")
    key = pd.Timestamp(target_date).isoformat()
    if key == _last_scaler_key:
        return _last_weighted_matrix

    fit_mask = (~np.isnat(candidate_release_dates)) & (candidate_release_dates < target_date)
    if not fit_mask.any():
        raise ValueError("No earlier releases are available to fit the target's scaler.")
    lower = raw_feature_matrix[fit_mask].min(axis=0)
    upper = raw_feature_matrix[fit_mask].max(axis=0)
    span = upper - lower
    scaled = np.zeros_like(raw_feature_matrix)
    np.divide(raw_feature_matrix - lower, span, out=scaled, where=span != 0)
    scaled = np.clip(scaled, 0.0, 1.0)
    _last_weighted_matrix = scaled * feature_multipliers
    _last_scaler_key = key
    historical_scaler_states[key] = {
        "fit_rule": "game_release_date < target_game_release_date",
        "fit_game_count": int(fit_mask.sum()),
        "feature_min": dict(zip(feature_columns, lower.tolist())),
        "feature_max": dict(zip(feature_columns, upper.tolist())),
    }
    return _last_weighted_matrix


def weighted_distances_to(target_idx, candidate_indices):
    """Full-vector weighted cosine or weighted Euclidean distance."""
    if len(candidate_indices) == 0:
        return np.empty(0, dtype=np.float64)
    matrix = weighted_matrix_for(target_idx)
    target_vector = matrix[target_idx]
    candidate_vectors = matrix[candidate_indices]
    if method == "euclidean":
        # Equals sqrt(sum_g normalized_weight_g * mean_j(scaled_diff_j ** 2)).
        return np.linalg.norm(candidate_vectors - target_vector, axis=1)

    target_norm = np.linalg.norm(target_vector)
    candidate_norms = np.linalg.norm(candidate_vectors, axis=1)
    denominators = target_norm * candidate_norms
    similarities = np.zeros(len(candidate_indices), dtype=np.float64)
    np.divide(
        np.einsum("ij,j->i", candidate_vectors, target_vector), denominators,
        out=similarities, where=denominators > 0,
    )
    # Exact all-zero vectors are identical; zero versus nonzero has distance 1.
    similarities[(target_norm == 0) & (candidate_norms == 0)] = 1.0
    return np.clip(1.0 - np.clip(similarities, -1.0, 1.0), 0.0, 1.0)


def rank_candidates(target_idx, candidate_mask):
    """Calculate and sort eligible candidates for one target."""
    candidate_indices = np.flatnonzero(candidate_mask)

    if len(candidate_indices) == 0:
        return []

    candidate_distances = weighted_distances_to(
        target_idx,
        candidate_indices,
    )

    # Sort by distance, then game name to break ties.
    sort_order = np.lexsort((
        game_names[candidate_indices],
        candidate_distances,
    ))

    return list(zip(
        candidate_indices[sort_order],
        candidate_distances[sort_order],
    ))


def make_neighbor_list(ranked_candidates, matched_name_col, limit):
    """Build a neighbor list from validated and ranked candidates."""
    neighbors = []
    if limit <= 0:
        return neighbors

    for candidate_idx, distance in ranked_candidates:
        candidate = game_records[int(candidate_idx)]
        matched_name = candidate[matched_name_col]

        neighbors.append({
            "join_game_name_key": str(candidate["join_game_name_key"]),
            "matched_source_name": str(matched_name).strip(),
            "similarity_score": float(max(0.0, 1.0 - distance)),
            "distance": float(distance),
        })

        if len(neighbors) >= limit:
            break

    return neighbors


# --------------------------------------------------
# Find Sales, Rental, and Eilers neighbors independently
# --------------------------------------------------
neighbor_cutoff = np.datetime64(pd.Timestamp(MAX_NEIGHBOUR_DATE))
all_indices = np.arange(len(game_records))


def candidate_masks_for(target_idx):
    """Share production eligibility rules with the single-game preview."""
    empty = np.zeros(len(game_records), dtype=bool)
    if not eligible_targets[target_idx] or np.isnat(candidate_release_dates[target_idx]):
        return {"sales": empty.copy(), "rental": empty.copy(), "eilers": empty.copy()}

    # Every neighbor must have a usable Eilers match, plus the existing
    # age, date-cutoff, and self-exclusion requirements.
    latest_neighbor_date = np.datetime64(
        pd.Timestamp(candidate_release_dates[target_idx]) - pd.DateOffset(months=MIN_AGE_GAP_MONTHS)
    )
    base_eligible = (
        has_eilers_match
        & valid_eilers_names
        & ~np.isnat(candidate_release_dates)
        & (candidate_release_dates < candidate_release_dates[target_idx])
        & (candidate_release_dates <= latest_neighbor_date)
        & (all_indices != target_idx)
        & (candidate_release_dates > neighbor_cutoff)
    )
    # P1 only restricts targets that have a parent reference. Candidates
    # need a parent of their own; it does not have to be the same parent.
    if PARENT_GAME_PROCESS == "p1_parent_games_only" and has_parent_game[target_idx]:
        base_eligible &= has_parent_game

    return {
        "sales": base_eligible & has_sales_match & valid_sales_names,
        "rental": base_eligible & has_rental_match & valid_rental_names,
        # Eilers-only games can still supply performance neighbors.
        "eilers": base_eligible,
    }


def ranked_candidates_for(target_idx, source, candidate_mask):
    """Apply the selected process; production and preview share this list."""
    ranked = rank_candidates(target_idx, candidate_mask)
    if (
        PARENT_GAME_PROCESS != "p3_include_parent"
        or source not in {"sales", "rental"}
        or not eligible_targets[target_idx]
    ):
        return ranked

    parent_idx, _ = parent_resolutions[target_idx]
    if parent_idx is None:
        return ranked
    available = (
        has_sales_match[parent_idx] and valid_sales_names[parent_idx]
        if source == "sales"
        else has_rental_match[parent_idx] and valid_rental_names[parent_idx]
    )
    if not available:
        return ranked
    # P3 may override the usual age gap/cutoff/Eilers requirement, but
    # a parent with an unknown or non-earlier release is never historical.
    if (np.isnat(candidate_release_dates[target_idx])
        or np.isnat(candidate_release_dates[parent_idx])
        or candidate_release_dates[parent_idx] >= candidate_release_dates[target_idx]):
        return ranked

    # Explicit parent inclusion is an exception to the usual candidate
    # age-gap, release cutoff and Eilers requirements. The parent must be
    # a distinct, uniquely resolved game with a usable commercial match.
    # Reserve one of the configured N slots, keeping its real distance.
    parent_distance = float(weighted_distances_to(
        target_idx, np.array([parent_idx], dtype=int)
    )[0])
    return [(parent_idx, parent_distance)] + [
        (index, distance) for index, distance in ranked if int(index) != parent_idx
    ]


def parent_is_in_list(target_idx, neighbors):
    parent_idx, _ = parent_resolutions[target_idx]
    return parent_idx is not None and any(
        item["join_game_name_key"] == game_names[parent_idx] for item in neighbors
    )


result_rows = []

for target_idx in np.flatnonzero(eligible_targets):
    target_idx = int(target_idx)
    target = game_records[target_idx]
    masks = candidate_masks_for(target_idx)

    # Compute all three lists for each eligible target. Match flags describe
    # the target's actual source availability, independently of these lists.
    sales_neighbors = make_neighbor_list(
        ranked_candidates_for(target_idx, "sales", masks["sales"]),
        matched_name_col=sales_name_col,
        limit=N_SALES_NEIGHBORS,
    )
    rental_neighbors = make_neighbor_list(
        ranked_candidates_for(target_idx, "rental", masks["rental"]),
        matched_name_col=rental_name_col,
        limit=N_RENTAL_NEIGHBORS,
    )
    eilers_neighbors = make_neighbor_list(
        ranked_candidates_for(target_idx, "eilers", masks["eilers"]),
        matched_name_col=eilers_name_col,
        limit=N_EILERS_NEIGHBORS,
    )

    target_release_date = pd.to_datetime(
        target.get("game_release_date"),
        errors="coerce",
    )

    parent_idx, parent_status = parent_resolutions[target_idx]
    result_rows.append({
        "parent_game": parent_keys[target_idx],
        "parent_game_process": PARENT_GAME_PROCESS,
        "parent_game_match_status": parent_status,
        "resolved_parent_game_name": (
            None if parent_idx is None else str(game_names[parent_idx])
        ),
        "parent_in_sales_neighbors": parent_is_in_list(target_idx, sales_neighbors),
        "parent_in_rental_neighbors": parent_is_in_list(target_idx, rental_neighbors),
        "join_game_name": optional_string(target, "join_game_name_key"),
        "to_sales_name_match": optional_string(
            target,
            "to_sales_name_match",
        ),
        "to_rental_name_match": optional_string(target, rental_name_col),
        "to_eilers_ts_name_match": optional_string(target, eilers_name_col),
        "has_sales_match": bool(has_sales_match[target_idx]),
        "has_rental_match": bool(has_rental_match[target_idx]),
        "has_eilers_ts_match": bool(has_eilers_match[target_idx]),
        "sales_rental_match_type": sales_rental_type(target_idx),
        "game_category": optional_string(target, "game_category"),
        "own_status": optional_string(target, "own_status"),
        "parent_own_status": optional_string(target, "parent_own_status"),
        "game_classification": optional_string(
            target,
            "game_classification",
        ),
        "max_months_since_release": (
            None
            if pd.isna(target.get("max_months_since_release"))
            else float(target["max_months_since_release"])
        ),
        "sales_similar_games": sales_neighbors,
        "rental_similar_games": rental_neighbors,
        "eilers_similar_games": eilers_neighbors,
        "experiment_readable_name": EXPERIMENT_READABLE_NAME,
        "experiment_id": EXPERIMENT_ID,
        "distance_method": method,
        "game_release_date": (
            None
            if pd.isna(target_release_date)
            else target_release_date.date()
        ),
        "feature_group_weights": {
            group: float(weight)
            for group, weight in normalized_weights.items()
        },
    })


# --------------------------------------------------
# Create Spark output
# --------------------------------------------------
neighbor_schema = StructType([
    StructField("join_game_name_key", StringType(), True),
    StructField("matched_source_name", StringType(), True),
    StructField("similarity_score", DoubleType(), True),
    StructField("distance", DoubleType(), True),
])

output_schema = StructType([
    StructField("join_game_name", StringType(), True),
    StructField("parent_game", StringType(), True),
    StructField("parent_game_process", StringType(), False),
    StructField("parent_game_match_status", StringType(), False),
    StructField("resolved_parent_game_name", StringType(), True),
    StructField("parent_in_sales_neighbors", BooleanType(), False),
    StructField("parent_in_rental_neighbors", BooleanType(), False),
    StructField("to_sales_name_match", StringType(), True),
    StructField("to_rental_name_match", StringType(), True),
    StructField("to_eilers_ts_name_match", StringType(), True),
    StructField("has_sales_match", BooleanType(), False),
    StructField("has_rental_match", BooleanType(), False),
    StructField("has_eilers_ts_match", BooleanType(), False),
    StructField("sales_rental_match_type", StringType(), False),
    StructField("game_category", StringType(), True),
    StructField("own_status", StringType(), True),
    StructField("parent_own_status", StringType(), True),
    StructField("game_classification", StringType(), True),
    StructField("max_months_since_release", DoubleType(), True),
    StructField(
        "sales_similar_games",
        ArrayType(neighbor_schema),
        True,
    ),
    StructField(
        "rental_similar_games",
        ArrayType(neighbor_schema),
        True,
    ),
    StructField(
        "eilers_similar_games",
        ArrayType(neighbor_schema),
        True,
    ),
    StructField("experiment_readable_name", StringType(), True),
    StructField("experiment_id", StringType(), True),
    StructField("distance_method", StringType(), True),
    StructField("game_release_date", DateType(), True),
    StructField(
        "feature_group_weights",
        MapType(StringType(), DoubleType()),
        True,
    ),
])

like_games_ai_attr_df = spark.createDataFrame(
    result_rows,
    schema=output_schema,
)

print(f"Parent game process: {PARENT_GAME_PROCESS}")
print(
    "Parent resolution status:",
    pd.Series([row["parent_game_match_status"] for row in result_rows], dtype="string")
    .value_counts().to_dict(),
)
print(
    "Targets with parent in Sales / Rental lists:",
    sum(row["parent_in_sales_neighbors"] for row in result_rows),
    "/",
    sum(row["parent_in_rental_neighbors"] for row in result_rows),
)
print(f"Input games: {len(pdf)}")
print(f"Eligible output games: {len(result_rows)}")
print(f"Excluded without an Eilers match: {int((~has_eilers_match).sum())}")
print(
    "Excluded with Eilers but neither Sales nor Rental: "
    f"{int((has_eilers_match & ~has_sales_match & ~has_rental_match).sum())}"
)
print(
    "Eligible games by match type:",
    pd.Series([row["sales_rental_match_type"] for row in result_rows], dtype="string")
    .value_counts().to_dict(),
)
display(like_games_ai_attr_df)

GENERATION_CONFIG = {
    **copy.deepcopy(PREPARED_SETTINGS),
    "normalized_weights": {
        group: float(normalized_weights.get(group, 0.0)) for group in feature_groups
    },
    "feature_groups": copy.deepcopy(feature_groups),
    "active_feature_groups": copy.deepcopy(active_groups),
    "feature_columns": list(feature_columns),
    "scaling": {
        "method": "per_target_historical_minmax",
        "fit_population": "all source games released strictly before target release",
        "bounds": [0.0, 1.0], "clip_out_of_range": True,
        "constant_feature_transform": 0.0, "numeric_missing_value": 0.0,
        "nonfinite_numeric_policy": "treat_as_missing",
        "group_feature_multiplier": "sqrt(normalized_group_weight / feature_count)",
    },
    "eligibility": {
        "target": "eilers match AND (sales match OR rental match)",
        "normal_neighbor": "earlier release, required age gap, date cutoff, usable Eilers mapping",
        "grouping_restrictions": [],
        "p1": "targets with a parent require neighbors that also have a parent",
        "p3": "commercial parent first; overrides age gap/date cutoff/Eilers; must predate target",
        "source_observation_asof_filter": "requires updated notebook 06",
    },
    "deduplication": "canonical game key; alphabetical document then content hash; merge match flags/names",
    "tie_breaker": "distance ascending, canonical game key ascending",
}

def canonical_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False,
                      default=lambda item: item.isoformat())

GENERATION_CONFIG_JSON = canonical_json(GENERATION_CONFIG)
SIMILARITY_CONFIG_HASH = hashlib.sha256(GENERATION_CONFIG_JSON.encode()).hexdigest()
GENERATION_INPUT_JSON = pdf.to_json(orient="records", date_format="iso", double_precision=15)
GENERATION_RESULT_JSON = canonical_json(result_rows)
SIMILARITY_RESULT_HASH = hashlib.sha256(GENERATION_RESULT_JSON.encode()).hexdigest()
GENERATION_SNAPSHOT = {
    "experiment_id": EXPERIMENT_ID,
    "config_json": GENERATION_CONFIG_JSON,
    "config_hash": SIMILARITY_CONFIG_HASH,
    "input_json": GENERATION_INPUT_JSON,
    "result_json": GENERATION_RESULT_JSON,
    "result_hash": SIMILARITY_RESULT_HASH,
    "scalers": copy.deepcopy(historical_scaler_states),
}
print(f"Configuration hash: {SIMILARITY_CONFIG_HASH}")
print("Neighbor generation is ready. The final cell logs MLflow and saves history.")

In [0]:
target_name = "king khufu dawn"

# Use the same candidate masks as the production search above.
target_keys = (
    pdf["join_game_name_key"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)
target_key = " ".join(target_name.lower().split())
target_indices = np.flatnonzero(
    target_keys.eq(target_key).fillna(False).to_numpy()
)


def make_candidate_preview(target_idx, source, candidate_mask, matched_name_col, limit):
    rows = []
    for rank, (candidate_idx, distance) in enumerate(
        ranked_candidates_for(target_idx, source, candidate_mask), start=1
    ):
        candidate_idx = int(candidate_idx)
        candidate = game_records[candidate_idx]
        rows.append({
            "selection_order": rank,
            "selected_neighbor": rank <= max(0, limit),
            "parent_game": parent_keys[candidate_idx],
            "is_target_parent": candidate_idx == parent_resolutions[target_idx][0],
            "join_game_name_key": candidate["join_game_name_key"],
            "matched_source_name": candidate[matched_name_col],
            "has_sales_match": bool(has_sales_match[candidate_idx]),
            "has_rental_match": bool(has_rental_match[candidate_idx]),
            "game_release_date": candidate["game_release_date"],
            "max_months_since_release": candidate_ages[candidate_idx],
            "months_older_than_target": (
                (pd.Timestamp(candidate_release_dates[target_idx]).year - pd.Timestamp(candidate_release_dates[candidate_idx]).year) * 12
                + pd.Timestamp(candidate_release_dates[target_idx]).month
                - pd.Timestamp(candidate_release_dates[candidate_idx]).month
                if not np.isnat(candidate_release_dates[target_idx])
                and not np.isnat(candidate_release_dates[candidate_idx]) else None
            ),
            "distance": float(distance),
            "similarity_score": float(max(0.0, 1.0 - distance)),
        })
    return pd.DataFrame(rows)


# Clear previews on every run so a skipped target cannot leave stale results.
sales_close_games_df = pd.DataFrame()
rental_close_games_df = pd.DataFrame()
eilers_close_games_df = pd.DataFrame()

if len(target_indices) != 1:
    print(f"Preview skipped: found {len(target_indices)} matches for {target_name!r}.")
else:
    target_idx = int(target_indices[0])
    if not eligible_targets[target_idx]:
        print(
            f"{target_name!r} is excluded from the output: "
            "an Eilers match and at least one Sales/Rental match are required."
        )
    else:
        print(f"Target: {target_name}; match type: {sales_rental_type(target_idx)}")
        print(f"Parent process: {PARENT_GAME_PROCESS}")
        print(f"Parent: {parent_keys[target_idx]!r}; status: {parent_resolutions[target_idx][1]}")
        masks = candidate_masks_for(target_idx)
        sales_close_games_df = make_candidate_preview(
            target_idx, "sales", masks["sales"], sales_name_col, N_SALES_NEIGHBORS
        )
        rental_close_games_df = make_candidate_preview(
            target_idx, "rental", masks["rental"], rental_name_col, N_RENTAL_NEIGHBORS
        )
        eilers_close_games_df = make_candidate_preview(
            target_idx, "eilers", masks["eilers"], eilers_name_col, N_EILERS_NEIGHBORS
        )
        for label, preview in [
            ("Sales", sales_close_games_df),
            ("Rental", rental_close_games_df),
            ("Eilers", eilers_close_games_df),
        ]:
            print(f"{label} candidates in selection order for {target_name} (selected_neighbor marks the saved list):")
            if preview.empty:
                print("No candidates meet the neighbor requirements.")
            else:
                display(preview)

In [0]:
# Save the frozen generation using the explicitly selected write mode.
# Overwrite replaces all table rows; append preserves prior generations.
import platform
import time
import mlflow
from mlflow.tracking import MlflowClient
from mlflow.entities import Param, Metric

if similarity_settings() != PREPARED_SETTINGS:
    raise ValueError("Settings changed. Run the notebook from setup to create a fresh experiment ID.")
if GENERATION_SNAPSHOT["experiment_id"] != EXPERIMENT_ID:
    raise ValueError("Run neighbor generation after setup before saving.")
if (canonical_json(GENERATION_CONFIG) != GENERATION_SNAPSHOT["config_json"]
    or canonical_json(result_rows) != GENERATION_SNAPSHOT["result_json"]):
    raise ValueError("Generated configuration/results changed. Rerun setup and generation.")
if not result_rows:
    raise ValueError("No eligible targets to save. Review source matches and configuration.")

config_hash = GENERATION_SNAPSHOT["config_hash"]
result_hash = GENERATION_SNAPSHOT["result_hash"]
save_identity = (EXPERIMENT_ID, config_hash, result_hash)
if SAVE_IDENTITY is not None and SAVE_IDENTITY != save_identity:
    raise ValueError("This experiment ID already belongs to different results. Rerun setup.")

existing_run_id = None
already_saved = False
if RESULT_WRITE_MODE == "append" and spark.catalog.tableExists(like_games_dest):
    previous = spark.table(like_games_dest).where(F.col("experiment_id") == EXPERIMENT_ID)
    previous_count = previous.count()
    if previous_count:
        needed = {"similarity_mlflow_run_id", "similarity_config_hash", "similarity_result_hash"}
        if not needed.issubset(previous.columns):
            raise ValueError("This experiment ID exists without lineage. Create a new experiment ID.")
        identities = previous.select(*sorted(needed)).distinct().collect()
        if (previous_count != len(result_rows) or len(identities) != 1
            or identities[0]["similarity_config_hash"] != config_hash
            or identities[0]["similarity_result_hash"] != result_hash
            or not identities[0]["similarity_mlflow_run_id"]):
            raise ValueError("This experiment ID already has different or incomplete saved results.")
        existing_run_id = identities[0]["similarity_mlflow_run_id"]
        already_saved = True

client = MlflowClient()
experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT_PATH)
if experiment is None:
    try:
        mlflow_experiment_id = client.create_experiment(MLFLOW_EXPERIMENT_PATH)
    except Exception:
        # Handle concurrent creation without hiding permission or network errors.
        experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT_PATH)
        if experiment is None:
            raise
        mlflow_experiment_id = experiment.experiment_id
else:
    mlflow_experiment_id = experiment.experiment_id

if existing_run_id:
    SIMILARITY_MLFLOW_RUN_ID = existing_run_id
if SIMILARITY_MLFLOW_RUN_ID is None:
    run = client.create_run(
        mlflow_experiment_id,
        tags={
            "mlflow.runName": f"{EXPERIMENT_READABLE_NAME}_{EXPERIMENT_ID[-8:]}",
            "pipeline_stage": "similarity",
            "similarity_experiment_id": EXPERIMENT_ID,
            "similarity_config_hash": config_hash,
            "implementation_version": IMPLEMENTATION_VERSION,
        },
    )
    SIMILARITY_MLFLOW_RUN_ID = run.info.run_id
SAVE_IDENTITY = save_identity
run_id = SIMILARITY_MLFLOW_RUN_ID

try:
    # Build from the frozen generation, preserving Spark's original date schema.
    saved_rows = json.loads(GENERATION_SNAPSHOT["result_json"])
    for row in saved_rows:
        if row["game_release_date"] is not None:
            row["game_release_date"] = pd.Timestamp(row["game_release_date"]).date()
    save_df = (
        spark.createDataFrame(saved_rows, schema=output_schema)
        .withColumn("target_game_key", F.col("join_game_name"))
        .withColumn("similarity_mlflow_run_id", F.lit(run_id))
        .withColumn("similarity_config_hash", F.lit(config_hash))
        .withColumn("similarity_result_hash", F.lit(result_hash))
        .withColumn("source_table_version", F.lit(RESOLVED_SOURCE_VERSION).cast("long"))
    )

    params = {
        "distance_method": method, "parent_game_process": PARENT_GAME_PROCESS,
        "result_write_mode": RESULT_WRITE_MODE,
        "neighbor_released_after": MAX_NEIGHBOUR_DATE,
        "minimum_age_gap_months": MIN_AGE_GAP_MONTHS,
        "n_sales_neighbors": N_SALES_NEIGHBORS, "n_rental_neighbors": N_RENTAL_NEIGHBORS,
        "n_eilers_neighbors": N_EILERS_NEIGHBORS,
        "source_table": ai_attr_silver_path, "source_table_version": RESOLVED_SOURCE_VERSION,
        "scaling_method": "per_target_historical_minmax", "missing_numeric_value": 0,
        "similarity_config_hash": config_hash, "implementation_version": IMPLEMENTATION_VERSION,
    }
    for group in feature_groups:
        params[f"raw_weight_{group}"] = FEATURE_GROUP_WEIGHTS.get(group, 0.0)
        params[f"weight_{group}"] = normalized_weights.get(group, 0.0)
    metrics = {
        "source_games": len(pdf), "eligible_targets": len(result_rows),
        "active_feature_count": len(feature_columns),
    }
    for channel in ["sales", "rental", "eilers"]:
        counts = [len(row[f"{channel}_similar_games"]) for row in result_rows]
        metrics[f"{channel}_mean_neighbors"] = float(np.mean(counts))
        metrics[f"{channel}_targets_without_neighbors"] = sum(n == 0 for n in counts)
        metrics[f"{channel}_neighbor_coverage"] = sum(n > 0 for n in counts) / len(counts)
    timestamp_ms = int(time.time() * 1000)
    client.log_batch(
        run_id,
        params=[Param(key, str(value)) for key, value in params.items()],
        metrics=[Metric(key, float(value), timestamp_ms, 0) for key, value in metrics.items()],
        synchronous=True,
    )
    client.log_dict(run_id, json.loads(GENERATION_SNAPSHOT["config_json"]), "configuration.json")
    client.log_dict(run_id, GENERATION_SNAPSHOT["scalers"], "preprocessing/historical_scalers.json")
    client.log_dict(run_id, {"games": json.loads(GENERATION_SNAPSHOT["input_json"])}, "data/prepared_games.json")
    client.log_dict(run_id, {"targets": saved_rows}, "results/neighbors.json")
    client.log_dict(run_id, {
        "python": platform.python_version(), "numpy": np.__version__,
        "pandas": pd.__version__, "mlflow": mlflow.__version__,
        "spark": spark.version,
    }, "environment.json")

    commit_metadata = json.dumps({
        "experiment_id": EXPERIMENT_ID, "mlflow_run_id": run_id,
        "result_write_mode": RESULT_WRITE_MODE,
    })
    if RESULT_WRITE_MODE == "overwrite":
        print(f"Replacing all rows in {like_games_dest} with {len(saved_rows)} targets.")
        (save_df.write.format("delta").mode("overwrite")
         .option("overwriteSchema", "true")
         .option("userMetadata", commit_metadata)
         .saveAsTable(like_games_dest))
    elif not already_saved:
        # Atomic duplicate protection covers concurrent append retries.
        (save_df.write.format("delta").mode("append")
         .option("mergeSchema", "true")
         .option("txnAppId", f"similarity:{EXPERIMENT_ID}")
         .option("txnVersion", 0)
         .option("userMetadata", commit_metadata)
         .saveAsTable(like_games_dest))

    if RESULT_WRITE_MODE == "overwrite":
        table_count = spark.table(like_games_dest).count()
        if table_count != len(saved_rows):
            raise RuntimeError(f"Overwrite verification failed: table has {table_count} rows; expected {len(saved_rows)}.")

    persisted = spark.table(like_games_dest).where(F.col("experiment_id") == EXPERIMENT_ID)
    saved_count = persisted.count()
    if saved_count != len(saved_rows):
        raise RuntimeError(f"Saved {saved_count} rows; expected {len(saved_rows)} for {EXPERIMENT_ID}.")
    saved_identities = persisted.select(
        "similarity_mlflow_run_id", "similarity_config_hash", "similarity_result_hash"
    ).distinct().collect()
    if (len(saved_identities) != 1
        or saved_identities[0]["similarity_mlflow_run_id"] != run_id
        or saved_identities[0]["similarity_config_hash"] != config_hash
        or saved_identities[0]["similarity_result_hash"] != result_hash):
        raise RuntimeError("Saved identity differs from this run, possibly due to a concurrent save. Reload the saved experiment before retrying.")
    commit = (
        spark.sql(f"DESCRIBE HISTORY {quoted_table_name(like_games_dest)}")
        .where(F.get_json_object(F.col("userMetadata"), "$.experiment_id") == EXPERIMENT_ID)
        .agg(F.max("version").alias("v")).first()["v"]
    )
    client.log_dict(run_id, {
        "table": like_games_dest, "experiment_id": EXPERIMENT_ID,
        "delta_commit_version": None if commit is None else int(commit),
        "row_count": saved_count, "result_write_mode": RESULT_WRITE_MODE,
        "configuration_sha256": config_hash,
        "results_sha256": result_hash,
    }, "results/delta_location.json")
    client.set_tag(run_id, "neighbor_save_status", "saved")
    client.set_terminated(run_id, status="FINISHED")
except BaseException as error:
    try:
        client.set_tag(run_id, "neighbor_save_status", "failed")
        client.set_tag(run_id, "failure_type", type(error).__name__)
        client.set_terminated(run_id, status="KILLED" if isinstance(error, KeyboardInterrupt) else "FAILED")
    except Exception:
        pass  # Preserve the original error when the tracking server is unavailable.
    raise

like_games_ai_attr_df = save_df
print("Replaced the results table with this generation." if RESULT_WRITE_MODE == "overwrite"
      else "Already saved; no duplicate rows added." if already_saved
      else "Appended a new neighbor generation.")
print(f"Experiment ID for notebook 06: {EXPERIMENT_ID}")
print(f"Similarity MLflow run ID: {run_id}")
print(f"MLflow experiment: {MLFLOW_EXPERIMENT_PATH}")
print(f"Targets saved: {saved_count}")
print("Notebook 06 still needs its forecast-run history and as-of observation updates.")